In [11]:
# === ConvNeXt-Large (384) | Stratified 5-Fold OOF 평가 (train만 사용, test 없음) ===
from fastai.vision.all import *
from fastai.callback.tracker import SaveModelCallback, EarlyStoppingCallback
from timm import create_model
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score
from pathlib import Path
import numpy as np, pandas as pd, torch, random, gc

# -----------------------------
# 0) 기본 설정
# -----------------------------
SEED = 42
set_seed(SEED, reproducible=True)
try: torch.set_float32_matmul_precision('high')
except: pass

DATA_ROOT = Path('images/competition_training')   # ★ 네가 쓰던 학습 폴더
KNOWN_CLASSES = ['0','1','2','3','4']             # 클래스 고정
IMG_SIZE = 384
BS = 16
NUM_WORKERS = 8
EPOCHS = 12               # 네 기본 10에서 살짝 + (안정 수렴)
BASE_LR = 1e-4
PATIENCE = 4
N_FOLDS = 5               # 5-fold OOF
PREC = 4                  # 출력 소수점 자리

MODEL_NAME = 'convnext_large_in22k'
RUN_TAG = 'convnext_oof'  # 저장 파일 접두사

# -----------------------------
# 1) 데이터 목록/라벨 수집
# -----------------------------
items_all = get_image_files(DATA_ROOT, recurse=True)
labels_all = np.array([parent_label(p) for p in items_all])
assert set(labels_all).issubset(set(KNOWN_CLASSES)), "라벨이 KNOWN_CLASSES 밖에 있네요. 폴더 구조 확인!"

# -----------------------------
# 2) 유틸: 클래스 가중치 (훈련 파트에서만 계산)
# -----------------------------
def compute_class_weights_by_items(items, classes):
    cnt = {c:0 for c in classes}
    for p in items:
        c = parent_label(p)
        if c in cnt: cnt[c]+=1
    arr = np.array([max(1, cnt[c]) for c in classes], dtype=np.float32)
    inv = 1.0 / arr
    w = inv / inv.sum() * len(classes)  # 평균≈1 정규화
    return torch.tensor(w, dtype=torch.float32), cnt

# -----------------------------
# 3) OOF 준비 (예측/정답 저장)
# -----------------------------
oof_preds = np.empty(len(items_all), dtype=object)  # 예측 라벨 문자열
oof_true  = np.array([parent_label(p) for p in items_all], dtype=object)
oof_idx   = np.arange(len(items_all))

# -----------------------------
# 4) Stratified K-Fold 루프
# -----------------------------
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_reports = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels_all)), labels_all), 1):
    print(f"\n===== FOLD {fold}/{N_FOLDS} =====")
    train_items = [items_all[i] for i in tr_idx]
    valid_items = [items_all[i] for i in va_idx]

    class_weights, cnts = compute_class_weights_by_items(train_items, KNOWN_CLASSES)
    print(f"[fold {fold}] train counts = {cnts}")
    print(f"[fold {fold}] class weights = {class_weights.tolist()}")

    # DataBlock / DataLoaders (네 기본 증강 유지: Resize+aug_transforms)
    dblock = DataBlock(
        blocks=(ImageBlock, CategoryBlock(vocab=KNOWN_CLASSES)),
        get_items=lambda _: items_all,                  # 고정 리스트
        splitter=lambda _: (tr_idx.tolist(), va_idx.tolist()),
        get_y=parent_label,
        item_tfms=Resize(IMG_SIZE),
        batch_tfms=[*aug_transforms(size=IMG_SIZE),
                    Normalize.from_stats(*imagenet_stats)]
    )
    dls = dblock.dataloaders(None, bs=BS, num_workers=NUM_WORKERS, shuffle=True)

    # 모델/학습기
    n_out = len(KNOWN_CLASSES)
    model = create_model(MODEL_NAME, pretrained=True, num_classes=n_out)

    macro_f1 = F1Score(average='macro'); macro_f1.name = 'macro_f1'  # 모니터 이름 확정
    learn = Learner(
        dls,
        model,
        loss_func=CrossEntropyLossFlat(weight=class_weights.to(dls.device)),
        metrics=[macro_f1, accuracy],
        cbs=[
            SaveModelCallback(monitor='macro_f1', fname=f'{RUN_TAG}_fold{fold}_best'),
            EarlyStoppingCallback(monitor='macro_f1', patience=PATIENCE)
        ]
    ).to_fp16()

    # 학습
    learn.fine_tune(EPOCHS, base_lr=BASE_LR)

    # 베스트 로드 & 폴드 검증 평가
    learn.load(f'{RUN_TAG}_fold{fold}_best')
    val_loss, val_f1, val_acc = learn.validate()
    print(f"[fold {fold}] Val -> loss={float(val_loss):.{PREC}f}, F1(macro)={float(val_f1):.{PREC}f}, acc={float(val_acc):.{PREC}f}")

    # OOF 예측 저장
    with torch.no_grad():
        val_logits, _ = learn.get_preds(dl=dls.valid, with_decoded=False, act=None)
    val_pred_idx = val_logits.argmax(dim=1).cpu().numpy()
    idx2lbl = {i:c for i,c in enumerate(KNOWN_CLASSES)}
    val_pred_lbl = np.array([idx2lbl[i] for i in val_pred_idx], dtype=object)

    oof_preds[va_idx] = val_pred_lbl

    # 폴드 리포트
    report = classification_report(oof_true[va_idx], oof_preds[va_idx],
                                   target_names=KNOWN_CLASSES, digits=PREC, zero_division=0)
    print(f"\n[fold {fold}] OOF report\n{report}")
    fold_reports.append((fold, float(val_f1)))

    # 메모리 정리
    del learn, dls, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# -----------------------------
# 5) 전체 OOF 점수
# -----------------------------
oof_macro_f1 = f1_score(oof_true, oof_preds, average='macro', labels=KNOWN_CLASSES)
print("\n===== OOF SUMMARY =====")
for f, score in fold_reports:
    print(f"fold {f}: macro-F1={score:.{PREC}f}")
print(f"\nOOF macro-F1 = {oof_macro_f1:.{PREC}f}")

# OOF 리포트/CSV 저장
print("\n[OOF classification report]")
print(classification_report(oof_true, oof_preds, target_names=KNOWN_CLASSES, digits=PREC, zero_division=0))

oof_df = pd.DataFrame({
    'image': [items_all[i].relative_to(DATA_ROOT).as_posix() for i in range(len(items_all))],
    'true':  oof_true,
    'pred':  oof_preds
})
oof_df.to_csv('oof_pred_convnext_large_stratified.csv', index=False, encoding='utf-8-sig')
print("\nSaved: oof_pred_convnext_large_stratified.csv")



===== FOLD 1/5 =====
[fold 1] train counts = {'0': 4640, '1': 80, '2': 240, '3': 40, '4': 200}
[fold 1] class weights = [0.022984983399510384, 1.3331289291381836, 0.4443763494491577, 2.666257858276367, 0.5332515835762024]


/home/elicer/myenv/lib/python3.12/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_large_in22k to current convnext_large.fb_in22k.
  model = create_fn(


epoch,train_loss,valid_loss,macro_f1,accuracy,time
0,1.186755,0.843045,0.386605,0.821538,01:21


Better model found at epoch 0 with macro_f1 value: 0.38660544273169667.


epoch,train_loss,valid_loss,macro_f1,accuracy,time
0,0.692665,0.470766,0.514128,0.873077,01:58
1,0.744386,0.663395,0.466127,0.793846,01:57
2,0.740420,0.322598,0.557226,0.916923,01:59
3,0.650641,0.463324,0.496686,0.865385,01:58
4,0.501766,0.431272,0.502892,0.876923,01:58
5,0.274339,0.356314,0.569107,0.897692,01:58
6,0.189362,0.335280,0.577634,0.919231,01:56
7,0.126700,0.344969,0.578970,0.927692,01:57
8,0.050755,0.331686,0.594119,0.932308,01:58
9,0.029927,0.339727,0.585237,0.925385,01:57


...

epoch,train_loss,valid_loss,macro_f1,accuracy,time
0,0.729624,0.519579,0.571023,0.873077,01:05
1,0.719761,0.341073,0.525622,0.916154,01:04
2,0.749387,0.759340,0.500958,0.754615,01:04
3,0.649355,0.842120,0.396653,0.725385,01:04
4,0.451295,0.466488,0.539254,0.858462,01:05


Better model found at epoch 0 with macro_f1 value: 0.5710226647664766.
No improvement since epoch 0: early stopping


/home/elicer/myenv/lib/python3.12/site-packages/fastai/learner.py:59: UserWarning: Saved file doesn't contain an optimizer state.
  elif with_opt: warn("Saved file doesn't contain an optimizer state.")


Better model found at epoch 0 with macro_f1 value: 0.8730769157409668.
[fold 5] Val -> loss=0.5196, F1(macro)=0.5710, acc=0.8731



[fold 5] OOF report
              precision    recall  f1-score   support

           0     0.9821    0.8991    0.9388      1160
           1     0.2800    0.3500    0.3111        20
           2     0.2923    0.6333    0.4000        60
           3     0.8000    0.4000    0.5333        10
           4     0.5513    0.8600    0.6719        50

    accuracy                         0.8731      1300
   macro avg     0.5811    0.6285    0.5710      1300
weighted avg     0.9215    0.8731    0.8909      1300


===== OOF SUMMARY =====
fold 1: macro-F1=0.5941
fold 2: macro-F1=0.5879
fold 3: macro-F1=0.5656
fold 4: macro-F1=0.6621
fold 5: macro-F1=0.5710

OOF macro-F1 = 0.5961

[OOF classification report]
              precision    recall  f1-score   support

           0     0.9704    0.9597    0.9650      5800
           1     0.2927    0.2400    0.2637       100
           2     0.4624    0.5333    0.4954       300
           3     0.5185    0.5600    0.5385        50
           4     0.677